# A2 · Decisión ZAP (cielo)

**Spec:** [`docs/spec_A2_codex_sky_zap.md`](../docs/spec_A2_codex_sky_zap.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Decide y aplica (o descarta) la sustracción de cielo con ZAP.

| | |
|---|---|
| **Entrada** | Cubo reducido |
| **Salida (QC/productos)** | Cubo con cielo tratado (sin QC separado en este run) |
| **Consume aguas abajo** | A3, A4 |


## Qué es ZAP y por qué se necesita

**ZAP** (*Zurich Atmosphere Purge*, Soto et al. 2016) es una sustracción de **residuos de cielo** para MUSE basada en PCA. El pipeline (`muse_scipost subtract_sky`) ya resta un modelo de cielo, pero el **airglow** (líneas de OH y [O I] atmosféricas) es intenso y **varía en el tiempo** entre la exposición de ciencia y el modelo → suelen quedar **residuos de skylines**. ZAP construye una base PCA con los spaxels de **cielo** (con las fuentes enmascaradas) y elimina las componentes que describen esos residuos, dejando la señal astrofísica.

**Por qué importa aquí:** un residuo de skyline mal restado puede **imitar o contaminar** una línea espectral. Buscamos una línea débil de Hα en el compañero, así que el cielo residual es un contaminante de primer orden.

**El peligro (por qué NO se aplica a ciegas):** ZAP necesita suficientes spaxels de cielo *reales*. En el **campo diminuto de NFM**, con una estrella brillante y su compañero, la fracción de cielo es baja y las eigencomponentes pueden **absorber señal del compañero** — incluso *fabricar o borrar* una línea en Hα. Regla del proyecto: ante la duda, **no tocar la señal**.

**Decisión pre-registrada** (`musepipe.reduction.sky_zap.classify_zap_decision`), por métrica, no por juicio. `R` = RMS mediano en ventanas de skyline ÷ RMS mediano en continuo, medido en aperturas de cielo vacías:

| Condición | Decisión |
|---|---|
| `R ≤ 1.5` | **no necesario** → `zap_applied = False` |
| `R > 2.0` | **necesario** → `zap_applied = True` |
| `1.5 < R ≤ 2.0` | zona gris → checkpoint (no aplicar, preguntar) |
| fracción de cielo `< 0.25` | cielo insuficiente → checkpoint (no aplicar) |

Los parámetros de ZAP quedan en *default*; no se itera buscando 'el mejor resultado'.


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
bash scripts/sky_zap.sh
```


In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('_nbcommon.py')))
import _nbcommon as nb
RUN_ID = nb.resolve_run_id(None)
print('run =', RUN_ID)
print('dir =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## Resultados que llevaron a la conclusión

Métrica **M4 de cielo** del QC del cubo (`stages/stage00q_qc.json`) aplicada a la regla de decisión pre-registrada.


In [ ]:
qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
m4 = qc.get('m4_sky', {})
R = m4.get('R')
print(f'm4_sky.R = {R}   (RMS skyline / RMS continuo en aperturas vacías)   [status {m4.get("status")}]')

# Regla pre-registrada (sky_zap.classify_zap_decision): low=1.5, high=2.0
LOW, HIGH = 1.5, 2.0
if R is None:
    decision = 'sin dato'
elif R <= LOW:
    decision = 'not_needed  ->  zap_applied = False'
elif R > HIGH:
    decision = 'needed      ->  zap_applied = True'
else:
    decision = 'gray_zone   ->  checkpoint (no aplicar)'
print(f'Regla:  R <= {LOW} no necesario | R > {HIGH} necesario     =>     {decision}')

print()
print('Corroboración (nota del QC):')
print('  ', qc.get('note'))
print('M1/M2 se midieron del SKY_SPECTRUM cacheado (airglow, 32 exp,',
      qc.get('m1_wavelength', {}).get('n_measurements'), 'medidas) porque el')
print('cubo restado de cielo tiene <8 skylines usables.')


## Decisiones y notas
- La métrica M4 (`R`) y la caracterización del airglow viven en A4/`stage00q_qc.json`; la regla de decisión, en `musepipe/reduction/sky_zap.py`.


## Conclusión (registrada)

**Decisión: ZAP NO aplicado — `zap_applied = False` (`not_needed`).**

- **Fecha del análisis:** 2026-07-09 (QC A4/M4 sobre el cubo realineado, commit `700f009`); el cubo se redujo el 2026-07-08.
- **Datos:** cubo NFM-AO auto-reducido `cube_telcorr.fits` (OB 3444577, Prog 109.23B7.002, **7 exposiciones** MUSE.2022-09-01T00:36–02:03) + `SKY_SPECTRUM` cacheado (32 exposiciones) para caracterizar el airglow.
- **Evidencia:** `R = 0.547 ≤ 1.5` (umbral *no necesario*); el cubo restado de cielo tiene **<8 skylines usables** (residual al nivel de ruido). En el campo diminuto NFM, ZAP aportaría ~0 y arriesgaría absorber señal del compañero.
- **Estado M4 = yellow:** el residuo es bajo, pero la escasez de skylines hace la métrica menos robusta que en WFM. No bloqueante.
- **Para el paper:** registrar como decisión con su métrica (R=0.547), **no** como omisión. Nada es paper-válido hasta cerrar el A-block.
